# FaceSearch AI - Snapwelt - Gradio Edition v5
### ArcFace · InsightFace · MTCNN · FAISS · DBSCAN · Gradio
**Run all cells top-to-bottom. Click the `gradio.live` link at the end.**

**v5 fixes:**
- Cluster share (Email/WhatsApp/Telegram/ZIP) uses real Gradio components — fully working
- Each cluster = separate Accordion with header + photo grid + share tabs
- Active color on radio buttons & checkbox toggles
- No JS hacks, pure Gradio wiring

## Check GPU

In [1]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch to T4 GPU runtime!')

Thu May  7 15:47:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install

In [2]:
%%capture
!pip install -q opencv-python-headless mtcnn deepface
!pip install -q insightface onnxruntime-gpu
!pip install -q faiss-gpu
!pip install -q 'gradio>=4.0'
!pip install -q scikit-learn plotly pandas
print('Done')

## Verify

In [3]:
import importlib
for mod,name in [('cv2','OpenCV'),('insightface','InsightFace'),('mtcnn','MTCNN'),
                 ('deepface','DeepFace'),('faiss','FAISS'),('gradio','Gradio'),
                 ('sklearn','Scikit-learn'),('plotly','Plotly')]:
    try:
        m=importlib.import_module(mod)
        print(f'OK {name:<22} {getattr(m,"__version__","OK")}')
    except ImportError: print(f'MISSING {name}')

OK OpenCV                 4.13.0
OK InsightFace            0.7.3
OK MTCNN                  OK
OK DeepFace               0.0.99
MISSING FAISS
OK Gradio                 5.50.0
OK Scikit-learn           1.6.1
OK Plotly                 5.24.1


## Core Helpers

In [4]:
import numpy as np, cv2, os, zipfile, shutil, pickle, time, io, json
import pandas as pd, tempfile, smtplib, ssl, http.client, urllib.parse, urllib.request, base64
from pathlib import Path
from PIL import Image
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from sklearn.preprocessing import normalize

S = dict(db=[], faiss_index=None, model_loaded=False, model_pkg=None,
         model_type=None, results=[], search_time=0,
         cluster_labels=None)

CLUSTER_COLORS = ['#6c63ff','#43e97b','#ff6584','#f7971e','#58c4f5',
                  '#b44fff','#ff9a44','#00d2ff','#ff5e99','#00c9a7',
                  '#ffd700','#fa8231','#a29bfe','#fd79a8','#55efc4']

def cc(cid): return '#6b6b8a' if cid==-1 else CLUSTER_COLORS[cid%len(CLUSTER_COLORS)]

def hex2bgr(h):
    h=h.lstrip('#'); return (int(h[4:6],16),int(h[2:4],16),int(h[0:2],16))

def load_models():
    try:
        from insightface.app import FaceAnalysis
        app=FaceAnalysis(name='buffalo_l',providers=['CUDAExecutionProvider','CPUExecutionProvider'])
        app.prepare(ctx_id=0,det_size=(640,640)); return app,'insightface'
    except Exception as e:
        print(f'InsightFace: {e}')
        try:
            from mtcnn import MTCNN; from deepface import DeepFace
            return (MTCNN(),DeepFace),'deepface'
        except Exception as e2:
            print(f'DeepFace: {e2}'); return None,None

def get_emb_if(model,img):
    faces=model.get(img)
    if not faces: return None,None
    f=max(faces,key=lambda x:(x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]))
    return f.normed_embedding.astype(np.float32),[int(v) for v in f.bbox]

def get_emb_df(models,img):
    _,DeepFace=models
    try:
        r=DeepFace.represent(img,model_name='ArcFace',detector_backend='mtcnn',enforce_detection=True,align=True)
        emb=np.array(r[0]['embedding'],dtype=np.float32); emb/=np.linalg.norm(emb)+1e-8
        reg=r[0].get('facial_area',{})
        return emb,[reg.get('x',0),reg.get('y',0),reg.get('x',0)+reg.get('w',100),reg.get('y',0)+reg.get('h',100)]
    except: return None,None

def extract_emb(img):
    if S['model_type']=='insightface': return get_emb_if(S['model_pkg'],img)
    elif S['model_type']=='deepface':  return get_emb_df(S['model_pkg'],img)
    return None,None

def cosine_sim(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-8))
def draw_box(img,bbox,col=(108,99,255),t=2):
    x1,y1,x2,y2=[int(v) for v in bbox]; cv2.rectangle(img,(x1,y1),(x2,y2),col,t); return img
def crop_face(img,bbox,pad=0.15):
    x1,y1,x2,y2=[int(v) for v in bbox]; h,w=img.shape[:2]
    pw=int((x2-x1)*pad); ph=int((y2-y1)*pad)
    return img[max(0,y1-ph):min(h,y2+ph),max(0,x1-pw):min(w,x2+pw)]
def bgr2pil(img): return Image.fromarray(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
def pil2bgr(img): return cv2.cvtColor(np.array(img),cv2.COLOR_RGB2BGR)
def build_faiss(embs):
    try:
        import faiss; idx=faiss.IndexFlatIP(embs.shape[1]); idx.add(embs); return idx
    except: return None
def img_to_b64(img_bgr,q=85):
    _,enc=cv2.imencode('.jpg',img_bgr,[cv2.IMWRITE_JPEG_QUALITY,q])
    return base64.b64encode(enc.tobytes()).decode()

print('Core helpers ready')


Core helpers ready


## All Handler Functions

In [5]:
import gradio as gr

# ─── Sidebar helpers ──────────────────────────────────────────────────────────
def fn_init():
    if S['model_loaded']: return f'Already loaded: {S["model_type"].upper()}', db_html()
    pkg,mt=load_models()
    if pkg:
        S.update(model_pkg=pkg,model_type=mt,model_loaded=True)
        return f'OK {mt.upper()} ready', db_html()
    return 'No backend available.', db_html()

def db_html():
    n=len(S['db'])
    return (f'<div style="background:#1a1a26;border:1px solid #2e2e45;border-radius:12px;'
            f'padding:20px 12px;text-align:center;margin:4px 0;">'
            f'<div style="font-size:2.4rem;font-weight:700;color:#ff6584;font-family:monospace;">{n}</div>'
            f'<div style="font-size:0.65rem;color:#6b6b8a;text-transform:uppercase;letter-spacing:2px;">FACES INDEXED</div></div>')

def fn_clear_db(): S.update(db=[],faiss_index=None,results=[],cluster_labels=None); return 'Cleared.',db_html()
def fn_export():
    if not S['db']: return None
    buf=io.BytesIO(); pickle.dump({'db':S['db']},buf)
    p='/tmp/face_embeddings.pkl'; open(p,'wb').write(buf.getvalue()); return p
def fn_import(f,use_faiss):
    if f is None: return 'No file.',db_html()
    S['db']=pickle.load(open(f.name,'rb')).get('db',[])
    if use_faiss and S['db']: S['faiss_index']=build_faiss(np.array([r['emb'] for r in S['db']]))
    return f'Loaded {len(S["db"])} faces.',db_html()

# ─── Build database ───────────────────────────────────────────────────────────
def fn_build(multi_files,zip_file,method,max_dim,use_faiss,show_crop,progress=gr.Progress()):
    if not S['model_loaded']: return 'Initialize models first!',db_html(),[],''
    tmpdir=tempfile.mkdtemp(); paths=[]
    try:
        if method=='ZIP Archive':
            if zip_file is None: return 'Upload a ZIP.',db_html(),[],''
            with zipfile.ZipFile(zip_file.name,'r') as z: z.extractall(tmpdir)
            for ext in ['jpg','jpeg','png','bmp','webp','JPG','JPEG','PNG']:
                paths.extend(Path(tmpdir).rglob(f'*.{ext}'))
        else:
            if not multi_files: return 'Upload images.',db_html(),[],''
            paths=[Path(f.name) for f in multi_files]
        if not paths: return 'No images found.',db_html(),[],''
        new_recs=[]; failed=0; previews=[]; total=len(paths)
        for i,p in enumerate(paths):
            progress((i+1)/total,desc=f'Processing {p.name} ({i+1}/{total})')
            try:
                img=cv2.imread(str(p))
                if img is None: failed+=1; continue
                h,w=img.shape[:2]
                if max(h,w)>max_dim:
                    sc=max_dim/max(h,w); img=cv2.resize(img,(int(w*sc),int(h*sc)))
                emb,bbox=extract_emb(img)
                if emb is not None:
                    _,enc=cv2.imencode('.jpg',img,[cv2.IMWRITE_JPEG_QUALITY,85])
                    new_recs.append(dict(filename=p.name,emb=emb,bbox=bbox,img_bytes=enc.tobytes()))
                    if len(previews)<8:
                        d=img.copy()
                        if show_crop and bbox: d=crop_face(d,bbox)
                        elif bbox: draw_box(d,bbox)
                        if d.size>0: previews.append((bgr2pil(d),p.name))
                else: failed+=1
            except: failed+=1
        S['db'].extend(new_recs)
        if use_faiss and S['db']: S['faiss_index']=build_faiss(np.array([r['emb'] for r in S['db']]))
        cards=(f'<div style="display:flex;gap:12px;margin-top:16px;">'
               f'<div style="flex:1;background:#12121a;border:1px solid #252535;border-radius:12px;padding:20px;text-align:center;"><div style="font-size:2.5rem;font-weight:700;color:#43e97b;font-family:monospace;">{len(new_recs)}</div><div style="font-size:0.65rem;color:#6b6b8a;letter-spacing:2px;text-transform:uppercase;">INDEXED</div></div>'
               f'<div style="flex:1;background:#12121a;border:1px solid #252535;border-radius:12px;padding:20px;text-align:center;"><div style="font-size:2.5rem;font-weight:700;color:#ff6584;font-family:monospace;">{failed}</div><div style="font-size:0.65rem;color:#6b6b8a;letter-spacing:2px;text-transform:uppercase;">NO FACE</div></div>'
               f'<div style="flex:1;background:#12121a;border:1px solid #252535;border-radius:12px;padding:20px;text-align:center;"><div style="font-size:2.5rem;font-weight:700;color:#58c4f5;font-family:monospace;">{len(S["db"])}</div><div style="font-size:0.65rem;color:#6b6b8a;letter-spacing:2px;text-transform:uppercase;">TOTAL DB</div></div>'
               f'</div>')
        return f'Done! {len(new_recs)} indexed, {failed} failed.',db_html(),previews,cards
    finally: shutil.rmtree(tmpdir,ignore_errors=True)

# ─── Search ───────────────────────────────────────────────────────────────────
def fn_search(q_img,threshold,top_k,use_faiss,remove_dups,dup_thresh,show_crop,progress=gr.Progress()):
    if not S['db']:    return 'Database empty.',None,None,[],''
    if not S['model_loaded']: return 'Initialize models first.',None,None,[],''
    if q_img is None: return 'Upload a query image.',None,None,[],''
    img_bgr=pil2bgr(q_img)
    progress(0.1,desc='Detecting face...')
    q_emb,q_bbox=extract_emb(img_bgr)
    if q_emb is None: return 'No face detected. Use a clearer frontal photo.',None,None,[],''
    q_crop=None
    if q_bbox:
        qc=crop_face(img_bgr.copy(),q_bbox); q_crop=bgr2pil(qc) if qc.size>0 else None
    progress(0.3,desc='Searching...')
    t0=time.time(); db=S['db']
    if use_faiss and S['faiss_index'] is not None:
        import faiss
        qn=q_emb/(np.linalg.norm(q_emb)+1e-8)
        sc,ix=S['faiss_index'].search(qn.reshape(1,-1),min(top_k*2,len(db)))
        cands=[(float(s),db[i]) for s,i in zip(sc[0],ix[0]) if float(s)>=threshold]
    else:
        sims=[cosine_sim(q_emb,r['emb']) for r in db]
        cands=sorted([(s,db[i]) for i,s in enumerate(sims) if s>=threshold],key=lambda x:x[0],reverse=True)[:top_k*2]
    elapsed=time.time()-t0
    if remove_dups:
        filt=[]
        for s,r in cands:
            if not any(cosine_sim(r['emb'],k['emb'])>dup_thresh for _,k in filt): filt.append((s,r))
        cands=filt
    cands=cands[:top_k]; S['results']=cands
    if not cands: return f'No matches above {threshold:.2f}.',q_img,q_crop,[],''
    best_sc,best_rec=cands[0]
    bimg=cv2.imdecode(np.frombuffer(best_rec['img_bytes'],np.uint8),cv2.IMREAD_COLOR)
    bdisp=crop_face(bimg,best_rec['bbox']) if show_crop and best_rec.get('bbox') else bimg.copy()
    if not show_crop and best_rec.get('bbox'): draw_box(bdisp,best_rec['bbox'],(67,233,123),3)
    best_pil=bgr2pil(bdisp) if bdisp.size>0 else None
    gallery=[]
    for s,r in cands:
        img=cv2.imdecode(np.frombuffer(r['img_bytes'],np.uint8),cv2.IMREAD_COLOR)
        if img is None: continue
        d=crop_face(img,r['bbox']) if show_crop and r.get('bbox') else img.copy()
        if not show_crop and r.get('bbox'): draw_box(d,r['bbox'],(67,233,123),2)
        if d.size>0: gallery.append((bgr2pil(d),f'{s:.4f}  {r["filename"]}'))
    avg=np.mean([s for s,_ in cands]); bc='#43e97b' if best_sc>=0.7 else('#f7971e' if best_sc>=0.5 else'#ff6584')
    met=(f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin:8px 0 16px;">'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:10px 16px;text-align:center;"><div style="font-size:1.5rem;font-weight:700;color:#43e97b;font-family:monospace;">{len(cands)}</div><div style="font-size:0.6rem;color:#6b6b8a;text-transform:uppercase;">Matches</div></div>'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:10px 16px;text-align:center;"><div style="font-size:1.5rem;font-weight:700;color:{bc};font-family:monospace;">{best_sc:.3f}</div><div style="font-size:0.6rem;color:#6b6b8a;text-transform:uppercase;">Best Score</div></div>'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:10px 16px;text-align:center;"><div style="font-size:1.5rem;font-weight:700;color:#e8e8f0;font-family:monospace;">{avg:.3f}</div><div style="font-size:0.6rem;color:#6b6b8a;text-transform:uppercase;">Avg Score</div></div>'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:10px 16px;text-align:center;"><div style="font-size:1.5rem;font-weight:700;color:#58c4f5;font-family:monospace;">{elapsed*1000:.0f}ms</div><div style="font-size:0.6rem;color:#6b6b8a;text-transform:uppercase;">Search Time</div></div>'
         f'</div><div style="color:#6b6b8a;font-size:0.75rem;">Showing {len(cands)} results sorted by similarity</div>')
    return f'Found {len(cands)} matches  Best: {best_sc:.4f}',q_img,q_crop,gallery,met

def fn_dl_csv():
    if not S['results']: return None
    df=pd.DataFrame([{'filename':r['filename'],'similarity':f'{s:.4f}'} for s,r in S['results']])
    p='/tmp/face_search_results.csv'; df.to_csv(p,index=False); return p
def fn_dl_zip():
    if not S['results']: return None
    p='/tmp/matched_faces.zip'
    with zipfile.ZipFile(p,'w') as z:
        for s,r in S['results']: z.writestr(r['filename'],r['img_bytes'])
    return p

# ─── Analytics ────────────────────────────────────────────────────────────────
def fn_analytics():
    db=S['db']
    if not db: return '<p style="color:#6b6b8a;">Build a database first.</p>',None
    embs=np.array([r['emb'] for r in db]); norms=np.linalg.norm(embs,axis=1)
    exts=pd.Series([Path(r['filename']).suffix.lower() for r in db]).value_counts()
    idx='FAISS' if S['faiss_index'] else 'NumPy'; fnames=[r['filename'] for r in db]
    ext_rows=''.join(f'<tr><td style="color:#6b6b8a;padding:3px 8px;">{e}</td><td style="color:#e8e8f0;font-family:monospace;padding:3px 8px;">{c}</td></tr>' for e,c in exts.items())
    fn_rows=''.join(f'<tr><td style="padding:2px 4px;font-size:0.7rem;color:#6b6b8a;">{i+1}</td><td style="padding:2px 4px;font-size:0.7rem;color:#e8e8f0;font-family:monospace;">{fn}</td></tr>' for i,fn in enumerate(fnames[:30]))
    html=(f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:16px;">'
          f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:14px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#58c4f5;font-family:monospace;">{len(db)}</div><div style="font-size:0.6rem;color:#6b6b8a;">Total Faces</div></div>'
          f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:14px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#e8e8f0;font-family:monospace;">{embs.shape[1]}</div><div style="font-size:0.6rem;color:#6b6b8a;">Dims</div></div>'
          f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:14px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#43e97b;font-family:monospace;">{norms.mean():.3f}</div><div style="font-size:0.6rem;color:#6b6b8a;">Mean Norm</div></div>'
          f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:14px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#f7971e;font-family:monospace;">{idx}</div><div style="font-size:0.6rem;color:#6b6b8a;">Index</div></div>'
          f'</div><div style="display:flex;gap:20px;flex-wrap:wrap;">'
          f'<div><b style="color:#6c63ff;font-size:0.7rem;font-family:monospace;">EXTENSIONS</b><br><table style="border-collapse:collapse;margin-top:4px;">{ext_rows}</table></div>'
          f'<div><b style="color:#6c63ff;font-size:0.7rem;font-family:monospace;">EMBEDDING STATS</b><br><table style="border-collapse:collapse;margin-top:4px;">'
          f'<tr><td style="color:#6b6b8a;padding:3px 8px;">Mean</td><td style="color:#e8e8f0;font-family:monospace;padding:3px 8px;">{norms.mean():.4f}</td></tr>'
          f'<tr><td style="color:#6b6b8a;padding:3px 8px;">Std</td><td style="color:#e8e8f0;font-family:monospace;padding:3px 8px;">{norms.std():.4f}</td></tr>'
          f'<tr><td style="color:#6b6b8a;padding:3px 8px;">Min</td><td style="color:#e8e8f0;font-family:monospace;padding:3px 8px;">{norms.min():.4f}</td></tr>'
          f'<tr><td style="color:#6b6b8a;padding:3px 8px;">Max</td><td style="color:#e8e8f0;font-family:monospace;padding:3px 8px;">{norms.max():.4f}</td></tr>'
          f'</table></div>'
          f'<div><b style="color:#6c63ff;font-size:0.7rem;font-family:monospace;">INDEXED FILES (first 30)</b><br><table style="border-collapse:collapse;margin-top:4px;">{fn_rows}</table></div>'
          f'</div>')
    hm=None
    if len(db)<=50:
        import plotly.graph_objects as go
        sim=embs@embs.T; lbs=[r['filename'][:14] for r in db]
        fig=go.Figure(data=go.Heatmap(z=sim,x=lbs,y=lbs,colorscale='Plasma',zmin=0,zmax=1))
        fig.update_layout(paper_bgcolor='rgba(0,0,0,0)',plot_bgcolor='rgba(0,0,0,0)',
                          font=dict(color='#e8e8f0',size=9),height=400,margin=dict(l=5,r=5,t=30,b=5))
        hm=fig
    return html,hm

# ─── Clustering ───────────────────────────────────────────────────────────────
def fn_cluster(eps,min_samp,show_crop,max_show,progress=gr.Progress()):
    db=S['db']
    if not db: return 'Build database first.','',None,None,[]
    from sklearn.cluster import DBSCAN
    import plotly.graph_objects as go
    progress(0.1,desc='Running DBSCAN...')
    embs=np.array([r['emb'] for r in db]); embs_n=normalize(embs)
    labels=DBSCAN(eps=eps,min_samples=min_samp,metric='cosine').fit_predict(embs_n)
    S['cluster_labels']=labels.tolist()
    nc=len(set(labels))-(1 if -1 in labels else 0); nn=int((labels==-1).sum())
    dom=pd.Series(labels[labels!=-1]).value_counts() if (labels!=-1).any() else pd.Series(dtype=int)
    lg=int(dom.iloc[0]) if len(dom)>0 else 0
    met=(f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin:10px 0 16px;">'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:12px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#43e97b;font-family:monospace;">{nc}</div><div style="font-size:0.6rem;color:#6b6b8a;">Clusters</div></div>'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:12px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#ff6584;font-family:monospace;">{nn}</div><div style="font-size:0.6rem;color:#6b6b8a;">Noise</div></div>'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:12px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#58c4f5;font-family:monospace;">{lg}</div><div style="font-size:0.6rem;color:#6b6b8a;">Largest</div></div>'
         f'<div style="background:#12121a;border:1px solid #252535;border-radius:10px;padding:12px 20px;text-align:center;"><div style="font-size:1.8rem;font-weight:700;color:#f7971e;font-family:monospace;">{eps}</div><div style="font-size:0.6rem;color:#6b6b8a;">eps</div></div>'
         f'</div>')
    cs=pd.Series(labels).value_counts().sort_index()
    fig_bar=go.Figure(data=go.Bar(x=['Noise' if k==-1 else f'C{k}' for k in cs.index],y=cs.values,
        marker_color=[cc(k) for k in cs.index],text=cs.values,textposition='outside'))
    fig_bar.update_layout(paper_bgcolor='rgba(0,0,0,0)',plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color='#e8e8f0',size=11),height=300,margin=dict(l=10,r=10,t=10,b=10),showlegend=False,
        xaxis=dict(gridcolor='#252535',color='#6b6b8a'),yaxis=dict(gridcolor='#252535',color='#6b6b8a'))
    fig_pca=None
    if len(db)>=4:
        from sklearn.decomposition import PCA
        pts=PCA(n_components=2,random_state=42).fit_transform(embs_n)
        fig_pca=go.Figure()
        fnames=[r['filename'] for r in db]
        for cid in set(labels):
            m=labels==cid; col=cc(cid); lbl='Noise' if cid==-1 else f'Cluster {cid}'
            fig_pca.add_trace(go.Scatter(x=pts[m,0],y=pts[m,1],mode='markers',name=lbl,
                marker=dict(color=col,size=9,opacity=0.85,line=dict(color='#0a0a0f',width=1)),
                text=[fnames[i] for i,v in enumerate(m) if v],
                hovertemplate='<b>%{text}</b><br>x:%{x:.3f}<br>y:%{y:.3f}<extra></extra>'))
        fig_pca.update_layout(paper_bgcolor='rgba(0,0,0,0)',plot_bgcolor='rgba(18,18,26,1)',
            font=dict(color='#e8e8f0',size=11),height=460,margin=dict(l=10,r=10,t=10,b=10),
            legend=dict(bgcolor='rgba(18,18,26,0.9)',bordercolor='#252535',borderwidth=1),
            xaxis=dict(title='PCA Component 1',gridcolor='#1a1a26',color='#6b6b8a'),
            yaxis=dict(title='PCA Component 2',gridcolor='#1a1a26',color='#6b6b8a'))
    # Build per-cluster image galleries (list of (pil, caption) per cluster)
    progress(0.8,desc='Building cluster galleries...')
    real=[c for c in sorted(set(labels)) if c!=-1]
    noise_=[-1] if -1 in labels else []
    all_gallery=[]
    for cid in real+noise_:
        members=[db[i] for i,v in enumerate(labels==cid) if v]
        col=cc(cid)
        for rec in members[:int(max_show)]:
            img=cv2.imdecode(np.frombuffer(rec['img_bytes'],np.uint8),cv2.IMREAD_COLOR)
            if img is None: continue
            if show_crop and rec.get('bbox'): d=crop_face(img,rec['bbox'])
            else:
                d=img.copy()
                if rec.get('bbox'): draw_box(d,rec['bbox'],hex2bgr(col),2)
            if d.size>0:
                lbl='Noise' if cid==-1 else f'Cluster {cid}'
                all_gallery.append((bgr2pil(d),f'{lbl} | {rec["filename"]}'))
    progress(1.0)
    return f'Done: {nc} clusters, {nn} noise.',met,fig_bar,fig_pca,all_gallery

def fn_clear_cluster():
    S['cluster_labels']=None
    return 'Clusters cleared.','',None,None,[]

def fn_dl_cluster_csv():
    if not S['cluster_labels'] or not S['db']: return None
    df=pd.DataFrame({'Filename':[r['filename'] for r in S['db']],'Cluster':S['cluster_labels']})
    p='/tmp/face_clusters.csv'; df.to_csv(p,index=False); return p

# ─── Per-cluster share functions (called from Gradio accordions) ──────────────
def _zip_cluster(cid):
    labels=np.array(S['cluster_labels']); members=[S['db'][i] for i,v in enumerate(labels==cid) if v]
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',zipfile.ZIP_DEFLATED) as z:
        for r in members: z.writestr(r['filename'],r['img_bytes'])
    return buf.getvalue(),members

def fn_dl_cluster_zip(cid_str):
    if not S['cluster_labels'] or not S['db']: return None
    try: cid=int(cid_str)
    except: return None
    data,_=_zip_cluster(cid); p=f'/tmp/cluster_{cid}_faces.zip'
    open(p,'wb').write(data); return p

def fn_email(cid_str,sender,pw,to,body):
    if not S['cluster_labels']: return 'Run clustering first.'
    try: cid=int(cid_str)
    except: return 'Invalid cluster ID.'
    if not sender or not pw or not to: return 'Fill sender, password and recipient.'
    zdata,members=_zip_cluster(cid); zname=f'cluster_{cid}_faces.zip'
    try:
        msg=MIMEMultipart(); msg['From']=sender; msg['To']=to
        msg['Subject']=f'FaceSearch AI - Cluster {cid} ({len(members)} faces)'
        msg.attach(MIMEText(body,'plain'))
        zp=MIMEApplication(zdata,Name=zname); zp['Content-Disposition']=f'attachment; filename="{zname}"'
        msg.attach(zp)
        ctx=ssl.create_default_context()
        with smtplib.SMTP('smtp.gmail.com',587) as srv:
            srv.starttls(context=ctx); srv.login(sender,pw); srv.sendmail(sender,to,msg.as_string())
        return f'Email sent to {to} with {len(members)} images!'
    except smtplib.SMTPAuthenticationError: return 'Auth failed. Use a Gmail App Password (not your real password).'
    except Exception as e: return f'Failed: {e}'

def fn_whatsapp(cid_str,phone,apikey,msg_text):
    if not phone or not apikey: return 'Enter phone number and CallMeBot API key.'
    try:
        enc=urllib.parse.quote(msg_text); ph=phone.replace('+','').replace(' ','')
        url=f'https://api.callmebot.com/whatsapp.php?phone={ph}&text={enc}&apikey={apikey}'
        r=urllib.request.urlopen(url,timeout=15); resp=r.read().decode()
        return 'WhatsApp message sent!' if 'Message sent' in resp else f'Response: {resp[:300]}'
    except Exception as e: return f'Failed: {e}'

def fn_telegram(cid_str,token,chat_id,caption):
    if not S['cluster_labels']: return 'Run clustering first.'
    try: cid=int(cid_str)
    except: return 'Invalid cluster ID.'
    if not token or not chat_id: return 'Enter bot token and chat ID.'
    zdata,members=_zip_cluster(cid); zname=f'cluster_{cid}_faces.zip'
    try:
        bnd='FSBnd'; parts=[]
        parts.append((f'--{bnd}\r\nContent-Disposition: form-data; name="chat_id"\r\n\r\n{chat_id}\r\n').encode())
        parts.append((f'--{bnd}\r\nContent-Disposition: form-data; name="caption"\r\n\r\n{caption}\r\n').encode())
        parts.append((f'--{bnd}\r\nContent-Disposition: form-data; name="document"; filename="{zname}"\r\nContent-Type: application/zip\r\n\r\n').encode()+zdata+b'\r\n')
        parts.append((f'--{bnd}--\r\n').encode())
        body=b''.join(parts)
        conn=http.client.HTTPSConnection('api.telegram.org')
        conn.request('POST',f'/bot{token}/sendDocument',body=body,
                     headers={'Content-Type':f'multipart/form-data; boundary={bnd}'})
        rd=json.loads(conn.getresponse().read().decode())
        return f'ZIP sent to Telegram {chat_id}!' if rd.get('ok') else f'Telegram error: {rd.get("description","unknown")}'
    except Exception as e: return f'Failed: {e}'

print('All handlers defined.')


All handlers defined.


## Launch Gradio UI

In [8]:
import gradio as gr

CSS = (
    "@import url('https://fonts.googleapis.com/css2?family=Sora:wght@300;400;600;700"
    "&family=Space+Mono:wght@400;700&display=swap');"
    "body,.gradio-container,.main{background:#0a0a0f!important;color:#e8e8f0!important;font-family:'Sora',sans-serif!important;}"
    ".gradio-container{max-width:100%!important;padding:0!important;}"
    "#sidebar{background:#12121a!important;border-right:1px solid #252535!important;padding:0!important;min-height:100vh;}"
    "#sidebar-inner{padding:16px 12px;}"
    ".sbl{font-family:'Space Mono',monospace!important;font-size:0.65rem!important;color:#6c63ff!important;"
    "text-transform:uppercase!important;letter-spacing:2.5px!important;"
    "border-bottom:1px solid #252535!important;padding-bottom:5px!important;margin:14px 0 8px!important;}"
    "input[type=range]{accent-color:#ff6584!important;}"
    "input,textarea{background:#1a1a26!important;border:1px solid #252535!important;color:#e8e8f0!important;border-radius:8px!important;}"
    "label span{color:#e8e8f0!important;font-size:0.82rem!important;}"
    ".init-btn button,.process-btn button,.find-btn button,.run-cluster-btn button,.clear-cluster-btn button"
    "{background:linear-gradient(135deg,#6c63ff,#8b85ff)!important;color:#fff!important;border:none!important;"
    "width:100%!important;font-family:'Space Mono',monospace!important;}"
    ".clear-db-btn button{background:#1a1a26!important;border:1px solid #ff6584!important;color:#ff6584!important;}"
    ".tabs>.tab-nav{background:#12121a!important;border:none!important;border-bottom:1px solid #252535!important;padding:6px 12px 0!important;}"
    ".tabs>.tab-nav>button{color:#6b6b8a!important;font-family:'Space Mono',monospace!important;font-size:0.78rem!important;"
    "border:none!important;background:transparent!important;padding:8px 14px!important;border-radius:8px 8px 0 0!important;}"
    ".tabs>.tab-nav>button.selected{background:#6c63ff!important;color:#fff!important;}"
    ".tabitem{background:#0a0a0f!important;padding:20px!important;}"
    ".info-box{background:#1a1a26;border:1px solid #252535;border-radius:12px;padding:16px;font-size:0.82rem;color:#6b6b8a;line-height:1.9;}"
    ".info-box b{color:#e8e8f0;}"
    ".status-out textarea{background:#1a1a26!important;border:1px solid #252535!important;color:#43e97b!important;"
    "font-family:'Space Mono',monospace!important;font-size:0.78rem!important;}"
    "input[type=checkbox]{accent-color:#6c63ff!important;width:38px!important;height:20px!important;cursor:pointer!important;}"
    ".gr-form label:has(input[type=checkbox]:checked)>span{color:#6c63ff!important;font-weight:600!important;}"
    "input[type=radio]{accent-color:#6c63ff!important;cursor:pointer!important;}"
    ".gr-form label:has(input[type=radio]:checked)>span{color:#6c63ff!important;font-weight:700!important;}"
    ".gr-accordion{background:#1a1a26!important;border:1px solid #252535!important;border-radius:10px!important;margin:0 0 20px!important;}"
    ".gr-accordion>.label-wrap{background:#1a1a26!important;border-bottom:1px solid #252535!important;padding:12px 16px!important;}"
    ".gr-accordion>.label-wrap>span{color:#e8e8f0!important;font-size:0.82rem!important;}"
    ".share-tabs>.tab-nav>button{font-size:0.75rem!important;padding:6px 12px!important;}"
    ".share-tabs>.tab-nav>button.selected{background:#6c63ff!important;color:#fff!important;}"
    ".send-btn button{background:linear-gradient(135deg,#6c63ff,#8b85ff)!important;color:#fff!important;border:none!important;width:100%!important;padding:12px!important;font-size:0.88rem!important;}"
)

HERO = (
    '<div style="background:linear-gradient(135deg,#0a0a0f,#12121a,#0d0d1a);'
    'border-bottom:1px solid #252535;padding:28px 40px 20px;text-align:center;">'
    '<h1 style="font-family:Space Mono,monospace;font-size:2.4rem;font-weight:700;'
    'background:linear-gradient(135deg,#6c63ff,#ff6584,#43e97b);'
    '-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin:0;">snapwelt  See. Snap. Share.</h1>'
    '<p style="color:#6b6b8a;font-size:0.75rem;margin:8px 0 12px;text-transform:uppercase;letter-spacing:3px;">'
    'FACE RECOGNITION SYSTEM AND SMART IMAGE MANAGEMENT USING DEEP LEARNING</p>'
    '<div>'
    '<span style="background:rgba(108,99,255,.15);border:1px solid rgba(108,99,255,.4);border-radius:20px;'
    'padding:3px 12px;font-size:.68rem;font-family:Space Mono,monospace;color:#6c63ff;margin:2px;">ArcFace</span>'
    '<span style="background:rgba(108,99,255,.15);border:1px solid rgba(108,99,255,.4);border-radius:20px;'
    'padding:3px 12px;font-size:.68rem;font-family:Space Mono,monospace;color:#6c63ff;margin:2px;">InsightFace</span>'
    '<span style="background:rgba(108,99,255,.15);border:1px solid rgba(108,99,255,.4);border-radius:20px;'
    'padding:3px 12px;font-size:.68rem;font-family:Space Mono,monospace;color:#6c63ff;margin:2px;">MTCNN</span>'
    '<span style="background:rgba(108,99,255,.15);border:1px solid rgba(108,99,255,.4);border-radius:20px;'
    'padding:3px 12px;font-size:.68rem;font-family:Space Mono,monospace;color:#6c63ff;margin:2px;">FAISS</span>'
    '<span style="background:rgba(108,99,255,.15);border:1px solid rgba(108,99,255,.4);border-radius:20px;'
    'padding:3px 12px;font-size:.68rem;font-family:Space Mono,monospace;color:#6c63ff;margin:2px;">DBSCAN</span>'
    '<span style="background:rgba(108,99,255,.15);border:1px solid rgba(108,99,255,.4);border-radius:20px;'
    'padding:3px 12px;font-size:.68rem;font-family:Space Mono,monospace;color:#6c63ff;margin:2px;">Gradio</span>'
    '</div></div>'
)

SL = lambda t,ic='': f'<div class="sbl">{ic} {t}</div>'

# ── How many clusters to pre-build Gradio accordions for
MAX_CLUSTER_ACCORDIONS = 20

with gr.Blocks(css=CSS, title='FaceSearch AI - Snapwelt') as demo:
    gr.HTML(HERO)

    with gr.Row(equal_height=True):

        # ══════════════════════════════════════════════════
        # LEFT SIDEBAR
        # ══════════════════════════════════════════════════
        with gr.Column(scale=1, min_width=260, elem_id='sidebar'):
            with gr.Column(elem_id='sidebar-inner'):
                gr.HTML(SL('SYSTEM','⚙'))
                with gr.Column(elem_classes='init-btn'):
                    init_btn = gr.Button('🚀 Initialize Models')
                init_out = gr.Textbox(show_label=False, interactive=False,
                                      placeholder='Click to load ArcFace...', elem_classes='status-out', lines=1)

                gr.HTML(SL('DATABASE','🗄'))
                db_html_out = gr.HTML(db_html())
                with gr.Column(elem_classes='clear-db-btn'):
                    clear_db_btn = gr.Button('🗑 Clear Database', size='sm')
                export_btn = gr.Button('💾 Export Embeddings (.pkl)', size='sm')
                export_file = gr.File(label='', interactive=False, height=50)

                gr.HTML(SL('SEARCH SETTINGS','🔧'))
                threshold  = gr.Slider(0.0,1.0,0.45,step=0.01,label='Match Threshold')
                dup_thresh = gr.Slider(0.8,1.0,0.95,step=0.01,label='Duplicate Threshold')
                remove_dups = gr.Checkbox(False, label='Remove Duplicates')
                max_results = gr.Slider(5,100,20,step=5,label='Max Results')
                use_faiss   = gr.Checkbox(True,  label='Use FAISS (fast search)')

                gr.HTML(SL('DISPLAY SETTINGS','🖼'))
                show_crop      = gr.Checkbox(True, label='Show Face Crops in Results')
                result_cols    = gr.Slider(2,6,4,step=1,label='Result Columns')
                show_score_bar = gr.Checkbox(True, label='Show Score Bar')

                gr.HTML(SL('IMPORT EMBEDDINGS','📂'))
                import_file    = gr.File(file_types=['.pkl'], label='Load .pkl', height=80)
                import_faiss_cb = gr.Checkbox(True, label='Rebuild FAISS after import')
                import_btn  = gr.Button('📥 Load Embeddings', size='sm')
                import_out  = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')

                gr.HTML('<div style="border-top:1px solid #252535;margin:14px 0 6px;"></div>')
                gr.HTML('<div style="font-family:monospace;font-size:0.6rem;color:#333;text-align:center;">FaceSearch AI · Gradio v5</div>')

        # ══════════════════════════════════════════════════
        # RIGHT MAIN
        # ══════════════════════════════════════════════════
        with gr.Column(scale=4):
            with gr.Tabs():

                # ── TAB 1: BUILD DATABASE ──────────────────
                with gr.TabItem('📁  BUILD DATABASE'):
                    with gr.Row():
                        with gr.Column(scale=3):
                            gr.HTML(SL('UPLOAD DATASET','📤'))
                            upload_method = gr.Radio(
                                ['Multiple Images','ZIP Archive'], value='Multiple Images',
                                label='Upload Method'
                            )
                            multi_imgs   = gr.File(file_count='multiple', file_types=['image'],
                                                   label='Upload Images', visible=True)
                            zip_file_inp = gr.File(file_count='single', file_types=['.zip'],
                                                   label='Upload ZIP Archive', visible=False)
                            def _toggle(m):
                                return gr.update(visible=m=='Multiple Images'), gr.update(visible=m=='ZIP Archive')
                            upload_method.change(_toggle, upload_method, [multi_imgs, zip_file_inp])
                            with gr.Row():
                                batch_sl   = gr.Slider(8,64,16,step=8,label='Batch Size')
                                max_dim_sl = gr.Slider(640,1920,1280,step=64,label='Max Image Dim')
                            with gr.Column(elem_classes='process-btn'):
                                build_btn = gr.Button('⚡ Process & Index Faces')
                            build_status = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')
                            build_cards  = gr.HTML()
                        with gr.Column(scale=2):
                            gr.HTML(SL('HOW IT WORKS','ℹ'))
                            gr.HTML('<div class="info-box"><b>Step 1:</b> Upload images or ZIP archive<br>'
                                    '<b>Step 2:</b> RetinaFace / MTCNN detects all faces<br>'
                                    '<b>Step 3:</b> ArcFace generates 512-d embeddings<br>'
                                    '<b>Step 4:</b> Embeddings stored in FAISS index<br>'
                                    '<b>Step 5:</b> Ready for sub-millisecond cosine search<br><br>'
                                    '<b>Supported formats:</b> JPG PNG BMP WEBP<br>'
                                    '<b>Tip:</b> Use T4 GPU on Colab for 10x speed<br><br>'
                                    '<b>Threshold Guide:</b><br>'
                                    '0.30-0.40 permissive (more results)<br>'
                                    '0.45 balanced (default)<br>'
                                    '0.55-0.70 strict (high confidence)</div>')
                    gr.HTML(SL('SAMPLE INDEXED FACES','🖼'))
                    build_gallery = gr.Gallery(label='', columns=8, height=200, show_label=False)

                # ── TAB 2: SEARCH FACES ────────────────────
                with gr.TabItem('🔍  SEARCH FACES'):
                    with gr.Row():
                        with gr.Column(scale=1, min_width=300):
                            gr.HTML(SL('QUERY IMAGE','🎯'))
                            with gr.Tabs():
                                with gr.TabItem('📁 Upload Image'):
                                    q_upload = gr.Image(type='pil', label='Upload a face image', image_mode='RGB')
                                with gr.TabItem('📸 Camera'):
                                    q_cam = gr.Image(type='pil', sources=['webcam'], label='Webcam capture', image_mode='RGB')
                            with gr.Column(elem_classes='find-btn'):
                                find_btn = gr.Button('🔍 Find Matches')
                            search_status = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')
                            gr.HTML('<div style="margin-top:10px;font-size:0.65rem;color:#6b6b8a;font-family:monospace;border-bottom:1px solid #252535;padding-bottom:4px;">📷 Uploaded Query</div>')
                            q_up_disp   = gr.Image(label='', height=190, show_label=False, interactive=False)
                            gr.HTML('<div style="margin-top:8px;font-size:0.65rem;color:#6b6b8a;font-family:monospace;border-bottom:1px solid #252535;padding-bottom:4px;">🔬 Cropped Face</div>')
                            q_crop_disp = gr.Image(label='', height=190, show_label=False, interactive=False)
                        with gr.Column(scale=3):
                            gr.HTML(SL('RESULTS','📋'))
                            metrics_html = gr.HTML()
                            with gr.Row():
                                with gr.Column():
                                    gr.HTML('<div style="font-size:0.72rem;color:#6b6b8a;text-align:center;font-family:monospace;">🎯 Query Face</div>')
                                    cmp_query = gr.Image(label='', height=280, show_label=False, interactive=False)
                                with gr.Column(scale=0, min_width=40):
                                    gr.HTML('<div style="display:flex;align-items:center;justify-content:center;height:280px;font-size:1.8rem;color:#6c63ff;">⟷</div>')
                                with gr.Column():
                                    gr.HTML('<div style="font-size:0.72rem;color:#6b6b8a;text-align:center;font-family:monospace;">🥇 Best Match</div>')
                                    cmp_best = gr.Image(label='', height=280, show_label=False, interactive=False)
                            gr.HTML(SL('ALL MATCHES','🖼'))
                            results_gallery = gr.Gallery(label='', columns=4, height=420, show_label=False)
                            with gr.Row():
                                dl_csv_btn = gr.Button('⬇ Download CSV', size='sm')
                                dl_zip_btn = gr.Button('📦 Download ZIP', size='sm')
                            dl_csv_file = gr.File(label='Results CSV', interactive=False, height=50)
                            dl_zip_file = gr.File(label='Matched ZIP',  interactive=False, height=50)

                # ── TAB 3: ANALYTICS ───────────────────────
                with gr.TabItem('📊  ANALYTICS'):
                    refresh_btn = gr.Button('🔄 Load Analytics', size='sm')
                    with gr.Row():
                        with gr.Column(scale=1): ana_html = gr.HTML()
                        with gr.Column(scale=1): heatmap  = gr.Plot(label='Pairwise Similarity Heatmap')

                    gr.HTML('<div style="border-top:1px solid #252535;margin:20px 0;"></div>')
                    gr.HTML(SL('FACE CLUSTERING (DBSCAN)','🔬'))
                    with gr.Row():
                        with gr.Column(scale=2):
                            eps_sl       = gr.Slider(0.1,0.9,0.4,step=0.05,label='DBSCAN eps (cosine distance)')
                            minsamp_sl   = gr.Slider(1,5,1,step=1,label='Min Samples per Cluster')
                            maxshow_sl   = gr.Slider(4,20,8,step=1,label='Max faces shown per cluster')
                            with gr.Row():
                                with gr.Column(elem_classes='run-cluster-btn'):
                                    run_cl_btn = gr.Button('▶ Run Clustering')
                                with gr.Column(elem_classes='clear-cluster-btn'):
                                    clr_cl_btn = gr.Button('✕ Clear Clusters')
                        with gr.Column(scale=1):
                            gr.HTML('<div class="info-box"><b>DBSCAN Clustering</b> groups similar faces automatically.<br><br>'
                                    '<b>eps guide:</b><br>0.20-0.30 strict<br>0.35-0.45 balanced<br>0.50-0.70 loose<br><br>'
                                    '<b>Cluster -1</b> = noise (unclassified faces)</div>')
                    cl_status = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')
                    cl_metrics = gr.HTML()
                    with gr.Row():
                        bar_plot = gr.Plot(label='Cluster Size Distribution')
                        pca_plot = gr.Plot(label='Embedding Space (PCA 2D)')

                    # CSV download
                    gr.HTML('<div style="border-top:1px solid #252535;margin:16px 0 8px;"></div>')
                    with gr.Row():
                        cl_csv_btn  = gr.Button('⬇ Download Full Cluster CSV', size='sm')
                        cl_csv_file = gr.File(label='Cluster CSV', interactive=False, height=50)

                    # ── VISUAL CLUSTER GALLERY + PER-CLUSTER SHARE ──
                    gr.HTML(SL('VISUAL CLUSTER GALLERY  (each cluster separate)','🖼'))
                    gr.HTML('<p style="color:#6b6b8a;font-size:0.78rem;margin:0 0 12px;">After running clustering, each cluster appears below with its photos and sharing options.</p>')

                    # Pre-build MAX_CLUSTER_ACCORDIONS accordions hidden by default.
                    # After clustering they get shown/updated dynamically.
                    cluster_accordion_components = []  # list of dicts per accordion
                    for ci in range(MAX_CLUSTER_ACCORDIONS):
                        with gr.Accordion(f'Cluster {ci}', open=False, visible=False) as acc:
                            cl_header_html = gr.HTML()
                            cl_gallery     = gr.Gallery(label='', columns=4, height=300, show_label=False)
                            # Share section inside accordion
                            gr.HTML('<div style="border-top:1px solid #252535;margin:12px 0 8px;"></div>')
                            gr.HTML('<p style="color:#e8e8f0;font-size:0.82rem;font-weight:600;margin:0 0 10px;">📤 Send directly via:</p>')
                            with gr.Tabs(elem_classes='share-tabs'):
                                # EMAIL
                                with gr.TabItem('📧 Email'):
                                    gr.HTML('<p style="color:#6b6b8a;font-size:0.78rem;margin:0 0 10px;">Enter your Gmail credentials below to send the ZIP file directly as an email attachment. Uses Gmail SMTP (port 587).</p>')
                                    em_cid    = gr.Textbox(value=str(ci), visible=False)
                                    em_sender = gr.Textbox(label='Your Gmail address', placeholder='you@gmail.com')
                                    em_pw     = gr.Textbox(label='Gmail App Password', type='password',
                                                           placeholder='xxxx xxxx xxxx xxxx')
                                    em_to     = gr.Textbox(label='Recipient email address', placeholder='friend@example.com')
                                    em_body   = gr.Textbox(label='Message (optional)', lines=3,
                                                           value=f'Hi,\n\nPlease find the face images from Cluster {ci} attached.\n\nSent via FaceSearch AI.')
                                    with gr.Column(elem_classes='send-btn'):
                                        em_btn = gr.Button('📧 Send Email with ZIP')
                                    em_status = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')
                                    em_btn.click(fn_email, inputs=[em_cid,em_sender,em_pw,em_to,em_body], outputs=em_status)

                                # WHATSAPP
                                with gr.TabItem('📱 WhatsApp'):
                                    gr.HTML('<p style="color:#6b6b8a;font-size:0.78rem;margin:0 0 10px;">Uses <b style="color:#e8e8f0;">CallMeBot API</b> (free). First send <code>I allow callmebot to send me messages</code> to <b style="color:#e8e8f0;">+34 644 59 77 39</b> on WhatsApp to activate your API key.</p>')
                                    wa_cid   = gr.Textbox(value=str(ci), visible=False)
                                    wa_phone = gr.Textbox(label='WhatsApp Number (with country code)', placeholder='+919876543210')
                                    wa_key   = gr.Textbox(label='CallMeBot API Key', placeholder='123456')
                                    wa_msg   = gr.Textbox(label='Message', lines=2,
                                                          value=f'Hi! Sharing face images from Cluster {ci} via FaceSearch AI.')
                                    with gr.Column(elem_classes='send-btn'):
                                        wa_btn = gr.Button('📱 Send WhatsApp Message')
                                    wa_status = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')
                                    wa_btn.click(fn_whatsapp, inputs=[wa_cid,wa_phone,wa_key,wa_msg], outputs=wa_status)

                                # TELEGRAM
                                with gr.TabItem('✈️ Telegram'):
                                    gr.HTML('<p style="color:#6b6b8a;font-size:0.78rem;margin:0 0 10px;">Uses <b style="color:#e8e8f0;">Telegram Bot API</b>. Create a bot via <b style="color:#e8e8f0;">@BotFather</b> → get token. Then get Chat ID from <code>api.telegram.org/bot&lt;TOKEN&gt;/getUpdates</code>.</p>')
                                    tg_cid   = gr.Textbox(value=str(ci), visible=False)
                                    tg_token = gr.Textbox(label='Telegram Bot Token', placeholder='123456:ABCdef...')
                                    tg_chat  = gr.Textbox(label='Chat ID', placeholder='numeric chat ID')
                                    tg_cap   = gr.Textbox(label='Caption (optional)', lines=2,
                                                          value=f'Cluster {ci} face images from FaceSearch AI 📁')
                                    with gr.Column(elem_classes='send-btn'):
                                        tg_btn = gr.Button('✈️ Send ZIP via Telegram')
                                    tg_status = gr.Textbox(show_label=False, interactive=False, lines=1, elem_classes='status-out')
                                    tg_btn.click(fn_telegram, inputs=[tg_cid,tg_token,tg_chat,tg_cap], outputs=tg_status)

                                # DOWNLOAD ZIP
                                with gr.TabItem('📦 Download ZIP'):
                                    dz_cid    = gr.Textbox(value=str(ci), visible=False)
                                    with gr.Column(elem_classes='send-btn'):
                                        dz_btn  = gr.Button('📦 Download Cluster ZIP')
                                    dz_file   = gr.File(label='Cluster ZIP', interactive=False, height=60)
                                    dz_btn.click(fn_dl_cluster_zip, inputs=[dz_cid], outputs=dz_file)

                        cluster_accordion_components.append({
                            'accordion': acc,
                            'header_html': cl_header_html,
                            'gallery': cl_gallery,
                        })

    # ── EVENT WIRING ─────────────────────────────────────────────────────────
    init_btn.click(fn_init, outputs=[init_out, db_html_out])
    clear_db_btn.click(fn_clear_db, outputs=[init_out, db_html_out])
    export_btn.click(fn_export, outputs=export_file)
    import_btn.click(fn_import, inputs=[import_file, import_faiss_cb], outputs=[import_out, db_html_out])

    build_btn.click(fn_build,
        inputs=[multi_imgs, zip_file_inp, upload_method, max_dim_sl, use_faiss, show_crop],
        outputs=[build_status, db_html_out, build_gallery, build_cards])

    def _search(qup, qcam, thr, topk, uf, rd, dt, sc):
        img = qup if qup is not None else qcam
        st,upl,crp,gal,met = fn_search(img,thr,topk,uf,rd,dt,sc)
        return st,upl,crp,gal,met,upl,crp
    find_btn.click(_search,
        inputs=[q_upload,q_cam,threshold,max_results,use_faiss,remove_dups,dup_thresh,show_crop],
        outputs=[search_status,q_up_disp,q_crop_disp,results_gallery,metrics_html,cmp_query,cmp_best])
    dl_csv_btn.click(fn_dl_csv, outputs=dl_csv_file)
    dl_zip_btn.click(fn_dl_zip, outputs=dl_zip_file)

    refresh_btn.click(fn_analytics, outputs=[ana_html, heatmap])

    # Clustering: update accordions dynamically
    def _cluster_and_update(eps, minsamp, show_crop, maxshow, progress=gr.Progress()):
        stat,met,bar,pca,gallery = fn_cluster(eps,minsamp,show_crop,maxshow,progress)
        if not S['cluster_labels']:
            # Hide all accordions
            updates = [stat, met, bar, pca]
            for _ in cluster_accordion_components:
                updates += [gr.update(visible=False), gr.HTML(''), []]
            return updates

        labels = np.array(S['cluster_labels']); db = S['db']
        real  = [c for c in sorted(set(labels)) if c!=-1]
        noise = [-1] if -1 in labels else []
        all_cids = real + noise

        updates = [stat, met, bar, pca]
        for i, comp in enumerate(cluster_accordion_components):
            if i < len(all_cids):
                cid   = all_cids[i]
                col   = cc(cid)
                lbl   = 'Noise (-1)' if cid==-1 else f'Cluster {cid}'
                num   = 'N' if cid==-1 else f'#{cid}'
                members = [db[j] for j,v in enumerate(labels==cid) if v]
                cnt   = len(members)
                # Build gallery for this cluster
                cl_gal = []
                for rec in members[:int(maxshow)]:
                    img = cv2.imdecode(np.frombuffer(rec['img_bytes'],np.uint8),cv2.IMREAD_COLOR)
                    if img is None: continue
                    if show_crop and rec.get('bbox'): d=crop_face(img,rec['bbox'])
                    else:
                        d=img.copy()
                        if rec.get('bbox'): draw_box(d,rec['bbox'],hex2bgr(col),2)
                    if d.size>0: cl_gal.append((bgr2pil(d), rec['filename']))
                header = (
                    f'<div style="display:flex;align-items:center;gap:12px;">'
                    f'<span style="font-family:Space Mono,monospace;font-size:1.1rem;font-weight:700;color:{col};">{num}</span>'
                    f'<span style="font-family:Space Mono,monospace;font-size:1.1rem;font-weight:700;color:{col};">{lbl}</span>'
                    f'<span style="background:rgba(108,99,255,0.2);border:1px solid rgba(108,99,255,0.4);'
                    f'border-radius:20px;padding:2px 10px;font-size:0.7rem;color:#6c63ff;font-family:monospace;">{cnt} faces</span>'
                    f'</div>'
                )
                updates += [
                    gr.update(visible=True, label=f'{lbl}  ({cnt} faces)', open=(i==0)),
                    gr.update(value=header),
                    cl_gal
                ]
            else:
                updates += [gr.update(visible=False), gr.update(value=''), []]
        return updates

    cluster_outputs = [cl_status, cl_metrics, bar_plot, pca_plot]
    for comp in cluster_accordion_components:
        cluster_outputs += [comp['accordion'], comp['header_html'], comp['gallery']]

    run_cl_btn.click(_cluster_and_update,
        inputs=[eps_sl, minsamp_sl, show_crop, maxshow_sl],
        outputs=cluster_outputs)

    def _clear_cluster():
        fn_clear_cluster()
        updates = ['Clusters cleared.', '', None, None]
        for _ in cluster_accordion_components:
            updates += [gr.update(visible=False), gr.update(value=''), []]
        return updates

    clr_cl_btn.click(_clear_cluster, outputs=cluster_outputs)
    cl_csv_btn.click(fn_dl_cluster_csv, outputs=cl_csv_file)

demo.queue().launch(share=True, server_port=7860, show_error=True, debug=False)

/tmp/ipykernel_1656/1958832899.py:70: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

## Optional: Quick Test

In [ ]:
from insightface.app import FaceAnalysis
import cv2, numpy as np
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider','CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640,640))
print('InsightFace ready!')

In [ ]:
from google.colab import files
from IPython.display import display
import PIL.Image, cv2, numpy as np
uploaded = files.upload()
for fname, data in uploaded.items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    faces = app.get(img)
    print(f'{fname}: {len(faces)} face(s)')
    for f in faces:
        b=[int(x) for x in f.bbox]; cv2.rectangle(img,b[:2],b[2:],(67,233,123),3)
    display(PIL.Image.fromarray(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)))

## Troubleshooting

| Issue | Fix |
|---|---|
| No GPU | Runtime → Change runtime type → **T4 GPU** |
| InsightFace error | Re-run install cell, restart runtime |
| faiss-gpu error | Change `faiss-gpu` to `faiss-cpu` in install cell |
| No face detected | Use a clear frontal face photo |
| Share not working | Check credentials; for Gmail use App Password not real password |